# AlphaGenesis - Getting Started

This notebook demonstrates the basic usage of AlphaGenesis for quantitative trading.

## Prerequisites

Make sure you have installed the package with:
```bash
poetry install
```

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from alphagenesis.data import DataFetcher
from alphagenesis.features import FeatureEngineer
from alphagenesis.risk import RiskManager, VaRCalculator
from alphagenesis.backtest import Backtester, PerformanceMetrics
from alphagenesis.utils import setup_logger

# Setup logging
setup_logger(log_level="INFO")

## 2. Generate Sample Data

For this demo, we'll generate sample OHLCV data. In production, you'd fetch this from WEEX API.

In [ ]:
# Generate sample data
dates = pd.date_range(end=datetime.now(), periods=1000, freq="1H")
price_base = 50000

np.random.seed(42)
returns = np.random.randn(1000) * 0.02  # 2% volatility
prices = price_base * (1 + returns).cumprod()

df = pd.DataFrame({
    "open": prices,
    "high": prices * 1.005,
    "low": prices * 0.995,
    "close": prices,
    "volume": np.random.uniform(1000, 5000, 1000),
}, index=dates)

print(f"Generated {len(df)} data points")
df.head()

## 3. Feature Engineering

Create technical indicators and features for analysis.

In [ ]:
feature_engineer = FeatureEngineer()
df_features = feature_engineer.create_features(df)

print(f"Created {len(df_features.columns)} total columns (including features)")
print("\nNew features added:")
new_features = [col for col in df_features.columns if col not in df.columns]
print(new_features[:10])  # Show first 10 features

## 4. Risk Assessment

Calculate VaR and assess portfolio risk.

In [ ]:
# Calculate returns
returns = df["close"].pct_change().dropna()

# Calculate VaR
var_calc = VaRCalculator()
var_metrics = var_calc.calculate_all_metrics(returns, confidence_level=0.95)

print("Risk Metrics (95% confidence):")
print(f"Historical VaR:    {var_metrics['historical_var']:.4f}")
print(f"Parametric VaR:    {var_metrics['parametric_var']:.4f}")
print(f"Monte Carlo VaR:   {var_metrics['monte_carlo_var']:.4f}")
print(f"Conditional VaR:   {var_metrics['conditional_var']:.4f}")

## 5. Define Trading Strategy

Simple moving average crossover strategy.

In [ ]:
def sma_crossover_strategy(data):
    """Simple moving average crossover strategy."""
    if len(data) < 50:
        return "hold"
    
    sma_20 = data["close"].rolling(20).mean()
    sma_50 = data["close"].rolling(50).mean()
    
    if len(sma_20) < 2 or len(sma_50) < 2:
        return "hold"
    
    # Buy signal: SMA20 crosses above SMA50
    if sma_20.iloc[-2] <= sma_50.iloc[-2] and sma_20.iloc[-1] > sma_50.iloc[-1]:
        return "buy"
    
    # Sell signal: SMA20 crosses below SMA50
    if sma_20.iloc[-2] >= sma_50.iloc[-2] and sma_20.iloc[-1] < sma_50.iloc[-1]:
        return "sell"
    
    return "hold"

## 6. Run Backtest

In [ ]:
# Initialize backtester
backtester = Backtester(
    initial_capital=100000,
    commission_rate=0.001,
    slippage_rate=0.0005,
)

# Run backtest
equity_curve = backtester.run(df, sma_crossover_strategy, symbol="BTC/USDT")
results = backtester.get_results()

print("\nBacktest Results:")
print(f"Initial Capital:   ${results['initial_capital']:,.2f}")
print(f"Final Equity:      ${results['final_equity']:,.2f}")
print(f"Total Return:      {results['total_return_pct']:.2f}%")
print(f"Total Trades:      {results['total_trades']}")

## 7. Calculate Performance Metrics

In [ ]:
# Calculate comprehensive metrics
equity_series = equity_curve["total_equity"]
metrics = PerformanceMetrics.calculate_all_metrics(equity_series)

print("\nPerformance Metrics:")
print(f"Annual Return:        {metrics['annual_return']*100:.2f}%")
print(f"Annual Volatility:    {metrics['annual_volatility']*100:.2f}%")
print(f"Sharpe Ratio:         {metrics['sharpe_ratio']:.2f}")
print(f"Sortino Ratio:        {metrics['sortino_ratio']:.2f}")
print(f"Calmar Ratio:         {metrics['calmar_ratio']:.2f}")
print(f"Max Drawdown:         {metrics['max_drawdown_pct']:.2f}%")

## 8. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Plot equity curve
plt.figure(figsize=(12, 6))
plt.plot(equity_curve.index, equity_curve["total_equity"], label="Equity")
plt.axhline(y=backtester.initial_capital, color='r', linestyle='--', label="Initial Capital")
plt.title("Equity Curve")
plt.xlabel("Date")
plt.ylabel("Portfolio Value ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Next Steps

1. **Configure WEEX API**: Add your API credentials to `.env` file
2. **Explore ML Models**: Try LSTM or Transformer models for prediction
3. **Advanced Strategies**: Implement more sophisticated trading strategies
4. **Risk Management**: Integrate position sizing and stop-loss management
5. **Portfolio Optimization**: Use multi-asset portfolio optimization
6. **Live Trading**: Test paper trading before going live